# Prototype for LLM-based fluency-oriented refiner
This notebook prototypes a LLM-based text refiner that modifies the input text such that it flows naturally and has no grammatical error.

Since the refiner's behavior is controlled by a prompt, this notebook includes functions to measure fluency and constraint-compliance of end outputs, such that the user of the notebook can compare the effect of different prompts for the refiner LLM.

In [107]:

### Imports

from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import importlib

from new_module.dev_utils.utils import read_outputs, ravel

import new_module.llm_experiments.refine_with_llm.utils
importlib.reload(new_module.llm_experiments.refine_with_llm.utils)
from new_module.llm_experiments.refine_with_llm.utils import check_prompt_repetition_modification, clean_generation

In [108]:

### Refiner Definition
import new_module.llm_experiments.refine_with_llm.refine_postprocessing
importlib.reload(new_module.llm_experiments.refine_with_llm.refine_postprocessing)
from new_module.llm_experiments.refine_with_llm.refine_postprocessing import TextRefiner

In [50]:
### Eval Function Definition

from evaluation.prompted_sampling.evaluate import fluency_classify, nli_score, toxicity_score

# fluency_classify(generations_df, output_file=None)
# nli_score(generations_df, write_file, device='cuda')
# toxicity_score(generations_df, perspective_file, perspective_rate_limit=5)

In [53]:
### Set Arguments
input_path = "outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0"
consider_prefix = True
SAMPLE_SIZE = 20

In [ ]:
### Define Refiner, Input
refiner = TextRefiner(consider_prefix=consider_prefix)

Loading model: Qwen/Qwen2.5-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [86]:
### Set Input Data
# read L&E outputs to refine (=input_data of this script)
input_data = read_outputs(input_path)
input_data = input_data.sample(SAMPLE_SIZE, random_state=42)

# create a deep copy of input_data and call it output_data
output_data = input_data.copy()
# output_data = input_data.sample(20)

In [110]:
### Set Prompt to Test

prompt_to_test = """### INSTRUCTIONS    
You are an expert editor. Given a prefix and its continuation, your task is to revise only the continuation so that it is grammatically correct and flows naturally from the prefix.
- Judge grammatical correctness and flow by combining the prefix and the continuation.
- Ensure the revised continuation flows naturally from the prefix.
- Avoid modifying the prefix.
- Avoid adding new information or removing existing information. 
- Avoid changing the intent, tone, or facts. 
- Avoid adding toxic, offensive, or harmful content.
- Only revise wording, grammar, or phrasing for clarity and naturalness. 
- If the continuation is already correct and natural when combined with the prefix, return n/a. 
- Only output the revised continuation or "n/a". 

### EXAMPLES
Prefix: One of the main challenges in this project
Original: They are coordinating multiple teams across different time zones.
Prefix: One of the main challenges in this project
Refined: is coordinating multiple teams across different time zones. 

### INPUT
Prefix: %s
Original: %s

### OUPUT
Prefix: %s
Refined: 
"""

### Update Refiner's Prompt Template
refiner.prompt_template = prompt_to_test
print(refiner.prompt_template)

### INSTRUCTIONS    
You are an expert editor. Given a prefix and its continuation, your task is to revise only the continuation so that it is grammatically correct and flows naturally from the prefix.
- Judge grammatical correctness and flow by combining the prefix and the continuation.
- Ensure the revised continuation flows naturally from the prefix.
- Avoid modifying the prefix.
- Avoid adding new information or removing existing information. 
- Avoid changing the intent, tone, or facts. 
- Avoid adding toxic, offensive, or harmful content.
- Only revise wording, grammar, or phrasing for clarity and naturalness. 
- If the continuation is already correct and natural when combined with the prefix, return n/a. 
- Only output the revised continuation or "n/a". 

### EXAMPLES
Prefix: One of the main challenges in this project
Original: They are coordinating multiple teams across different time zones.
Prefix: One of the main challenges in this project
Refined: is coordinating multiple te

In [119]:
# for each row of input_data, refine "text" value and save it into output_data
refined_texts = []
for i, row in tqdm(input_data.iterrows(), total = len(input_data)):
    refined_texts.append(refiner.refine(row['text'], prefix=row['prompt'] if consider_prefix else ""))
    
# format change to match L&E outputs
output_data['text'] = refined_texts

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:40<00:00,  2.02s/it]


In [120]:
### Check Raw Outputs

prompt_repeated_count_i, prompt_modified_count_i = check_prompt_repetition_modification(input_data, verbose=False)
prompt_repeated_count_o, prompt_modified_count_o = check_prompt_repetition_modification(output_data, verbose=False)
print('=' * 30)
print(f"[INPUT ] PROMPT REPEATED COUNT: {prompt_repeated_count_i}, PROMPT MODIFIED COUNT: {prompt_modified_count_i}")
print(f"[OUTPUT] PROMPT REPEATED COUNT: {prompt_repeated_count_o}, PROMPT MODIFIED COUNT: {prompt_modified_count_o}")


[INPUT ] PROMPT REPEATED COUNT: 0, PROMPT MODIFIED COUNT: 0
[OUTPUT] PROMPT REPEATED COUNT: 6, PROMPT MODIFIED COUNT: 1


In [121]:
### Clean Generations
output_data["text"] = output_data.apply(lambda x: clean_generation(x['prompt'], x['text']), axis=1)

In [122]:
### Re-check Outputs
prompt_repeated_count_i, prompt_modified_count_i = check_prompt_repetition_modification(input_data, verbose=False)
prompt_repeated_count_o, prompt_modified_count_o = check_prompt_repetition_modification(output_data, verbose=False)
print('=' * 30)
print(f"[INPUT ] PROMPT REPEATED COUNT: {prompt_repeated_count_i}, PROMPT MODIFIED COUNT: {prompt_modified_count_i}")
print(f"[OUTPUT] PROMPT REPEATED COUNT: {prompt_repeated_count_o}, PROMPT MODIFIED COUNT: {prompt_modified_count_o}")

[INPUT ] PROMPT REPEATED COUNT: 0, PROMPT MODIFIED COUNT: 0
[OUTPUT] PROMPT REPEATED COUNT: 0, PROMPT MODIFIED COUNT: 1


In [123]:
### If Satisfactory, Ravel Output Data

output_data_r = ravel(output_data)

In [124]:
### For Reference, Eval Input Data
input_data_r = ravel(input_data)
fluency_i = fluency_classify(input_data_r, None)
avg_max_toxicity_i, toxic_proba_p_i, avg_toxicity_i, toxic_proba_s_i = toxicity_score(input_data_r, "dummy.txt")

print(f"% of Fluent Generations: {fluency_i * 100:.2f} %")
print(f"Avg Max Toxicity: {avg_max_toxicity_i:.4f}")
print(f"% of Prompts with At Least 1 Toxic Generation: {toxic_proba_p_i * 100:.2f} %")
print(f"Avg Toxicity: {avg_toxicity_i:.4f}")
print(f"% of Toxic Generations: {toxic_proba_s_i * 100:.2f} %")

Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


writing outputs to  None


Evaluating toxicity via perspective API: 100%|████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:31<00:00,  1.56s/it]

% of Fluent Generations: 100.00 %
Avg Max Toxicity: 0.1053
% of Prompts with At Least 1 Toxic Generation: 0.00 %
Avg Toxicity: 0.1053
% of Toxic Generations: 0.00 %


In [125]:
### Eval
fluency = fluency_classify(output_data_r, None)
avg_max_toxicity, toxic_proba_p, avg_toxicity, toxic_proba_s = toxicity_score(output_data_r, "dummy.txt")

print(f"% of Fluent Generations: {fluency * 100:.2f} %")
print(f"Avg Max Toxicity: {avg_max_toxicity:.4f}")
print(f"% of Prompts with At Least 1 Toxic Generation: {toxic_proba_p * 100:.2f} %")
print(f"Avg Toxicity: {avg_toxicity:.4f}")
print(f"% of Toxic Generations: {toxic_proba_s * 100:.2f} %")

Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


writing outputs to  None


Evaluating toxicity via perspective API: 100%|████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:30<00:00,  1.51s/it]

% of Fluent Generations: 100.00 %
Avg Max Toxicity: 0.1365
% of Prompts with At Least 1 Toxic Generation: 0.00 %
Avg Toxicity: 0.1365
% of Toxic Generations: 0.00 %
